In [153]:
# supervisor

# wedding venue
# DJ - playlist 
# Flights

# tools
# - find wedding venues
# - find DJs
# - find flights

# prompt - You are a wedding planner coordinator. You can call subagents to help you with the planning.


In [154]:
from dotenv import load_dotenv

load_dotenv()

True

In [155]:
from langchain.tools import tool 
from typing import Dict, Any 
from tavily import TavilyClient

tavily_client = TavilyClient()

tools = []

@tool
def search(query: str) -> Dict[str, Any]:
    """Searches web for info"""
    return tavily_client.search(query)


In [156]:
from langchain.agents import AgentState

class WeddingPlannerState(AgentState):
    number_of_guests: int
    date: str
    origin: str 
    destination: str
    genre: str

In [157]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools = await client.get_tools()

In [158]:
from langchain.agents import create_agent

wedding_venue_agent = create_agent(
    model='gpt-5-nano',
    tools=[search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options. 
    You have a suggested limit of 12 web searches. Count every web_search call you make.
    After 12 searches, you should stop searching and summarize the best options you have
    found so far.
    """
)

dj_agent = create_agent(
    model='gpt-5-nano',
    tools=[search],
    system_prompt="""You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.

    This is a SQLite database. Before writing any data queries, first discover the schema.
    """
)

flights_agent = create_agent(
    model='gpt-5-nano',
    tools=[tools[-1]], 
    system_prompt="""You are a travel agent. Search for flights to the desired destination wedding location.
    You are not allowed to ask any more follow up questions, you must find the best flight options based on the following criteria:
    - Price (lowest, economy class)
    - Duration (shortest)
    - Date (time of year which you believe is best for a wedding at this location)
    To make things easy, only look for one ticket, one way.
    You may need to make multiple searches to iteratively find the best options.
    You will be given no extra information, only the origin and destination. It is your job to think critically about the best options.
    If the MCP tool fails, returns malformed output, or does not give you usable flight results, try the tool again.
    Once you have found the best options, let the user know your shortlist of options.
    """
)



In [ ]:
from langchain.messages import HumanMessage, ToolMessage
from langchain.tools import ToolRuntime

@tool
def call_wedding_venue_agent(runtime: ToolRuntime) -> str:
    """Call wedding venue subagent to find the best venue for the wedding given a location and capacity"""
    destination = runtime.state["destination"]
    capacity = runtime.state["number_of_guests"]
    query = f"Find the best venue for the wedding in {destination} for {capacity} guests"
    response = wedding_venue_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response["messages"][-1].content

@tool
def call_dj_agent(runtime: ToolRuntime) -> str:
    """Call dj subagent to ask to find a playlis for the given genre."""
    genre = runtime.state["genre"]
    query = f"Find a playlist for the wedding. Genre: {genre}"
    response = dj_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response["messages"][-1].content

@tool
async def call_flights_agent(runtime: ToolRuntime) -> str:
    """Call flights subagent to find the best flights for the wedding."""
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]
    query = f"Find the best flights for the wedding. Origin: {origin}, Destination: {destination}"
    response = await flights_agent.ainvoke({"messages": [HumanMessage(content=query)]})
    return response["messages"][-1].content


In [160]:
from langgraph.types import Command 

@tool
def update_state(origin: str, destination: str, number_of_guests: int, genre: str, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origin, destination, number_of_guests, genre. 
    This tool must be called alone, without any other tool calls. It must complete and return to make,
    the information available to other tools."""
    return Command(update={
        "origin": origin,
        "destination": destination,
        "number_of_guests": number_of_guests,
        "genre": genre,
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]
    })


In [164]:
from langgraph.checkpoint.memory import InMemorySaver

main_agent = create_agent(
    model='gpt-5-nano',
    tools=[
        call_wedding_venue_agent, 
        call_dj_agent, 
        call_flights_agent,
        update_state
    ],
    checkpointer=InMemorySaver(),
    system_prompt="""
        You are a wedding planning agent that can call subagents to delegate tasks -
        First find all info you need to update the state. 
        When you have the information, update the state.
        If you don't have all that information - tersely ask for the minimal information needed. 
        Only after that has completed and returned, you can delegate tasks to your specialists 
        for flights, venues, playlists. 
        After recieving thier answers, coordiante the perfect wedding for me.
    """,
    state_schema=WeddingPlannerState
)

In [166]:
config = {
    "configurable": {"thread_id": "99"},
    "tags": ["WP"],
    "recursion_limit": 50
}

response = await main_agent.ainvoke(
    {"messages": [HumanMessage(content="plan a wedding in San Francisco - we're flying in from london. 100 guests, jazz-genre. 11/6/2026-11/8/2026. Style: romantic modern")]},
    config
)

TypeError: 'coroutine' object is not subscriptable

In [ ]:
from pprint import pprint 
pprint(response)

{'messages': [HumanMessage(content='plan a wedding in San Francisco. 100 guests, jazz-genre. 11/6-11/8. Style: romantic modern', additional_kwargs={}, response_metadata={}, id='3d59bae8-03fc-4f93-8e91-f6f59369d63f'),
              AIMessage(content='Fantastic—planning a romantic modern wedding in San Francisco for 100 guests with a jazz vibe sounds wonderful. I can start outlining a detailed plan as soon as I have a couple of key details. Right now, I’m missing a few essentials to update the system state and to proceed with venue, music, and travel arrangements.\n\nPlease help confirm the following:\n- Year for the dates: 11/6-11/8 of which year? (2026 or 2027, etc.)\n- Origin for guests: Where are the guests traveling from, or should I treat most guests as local with some travel considerations?\n- Budget range: What is your rough total budget or per-category budgets (venue, catering, music, decor, etc.)?\n- Neighborhood or vibe preferences in SF: Any favorite areas (SoMa, Mission, Mar